# PySpark DataFrame DSL — Part 8: Temporary Views, SQL Queries, and SQL vs DSL

> **Goal:** Learn to register DataFrames as SQL views, query them using Spark SQL, and compare SQL with the DataFrame DSL using realistic ETL examples. This follows the earlier notes pattern: definitions, parameters, multiple examples, explanations, large-data practice, common mistakes, and a quick reference.

## Contents
1. Setup and sample data
2. Temporary views and `createOrReplaceTempView()`
3. Ten temporary-view examples
4. `spark.sql()` and ten query examples
5. SQL vs DSL: ten paired examples
6. Bronze–Silver–Gold ETL example
7. Large-data example and plan inspection
8. Scope, lifecycle, mistakes, performance
9. Assignments and quick reference

---

# 1. Setup and sample data

```python
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (SparkSession.builder
         .appName("TempViews_SQL_vs_DSL")
         .getOrCreate())

orders_data = [
    (1001, "C101", "Arun", "Chennai", "Laptop", "Electronics", 65000.0, 1, "COMPLETED", "2026-01-05"),
    (1002, "C102", "Priya", "Bengaluru", "Phone", "Electronics", 32000.0, 2, "COMPLETED", "2026-01-06"),
    (1003, "C103", "Rahul", "Mumbai", "Desk", "Furniture", 12000.0, 1, "PENDING", "2026-01-07"),
    (1004, "C101", "Arun", "Chennai", "Headphones", "Electronics", 2500.0, 2, "COMPLETED", "2026-01-08"),
    (1005, "C104", "Meena", "Hyderabad", "Chair", "Furniture", 7500.0, 4, "CANCELLED", "2026-01-09"),
    (1006, "C102", "Priya", "Bengaluru", "Tablet", "Electronics", 22000.0, 1, "COMPLETED", "2026-01-10"),
    (1007, "C105", "Vikram", "Chennai", "Desk", "Furniture", 14000.0, 2, "COMPLETED", "2026-01-11"),
    (1008, "C103", "Rahul", "Mumbai", "Phone", "Electronics", 30000.0, 1, "PENDING", "2026-01-12"),
    (1009, "C106", "Divya", "Pune", "Monitor", "Electronics", 18000.0, 2, "COMPLETED", "2026-01-13"),
    (1010, "C104", "Meena", "Hyderabad", "Table", "Furniture", 16000.0, 1, "COMPLETED", "2026-01-14"),
    (1011, "C107", "Kavin", "Chennai", "Keyboard", "Electronics", 1800.0, 3, "COMPLETED", "2026-01-15"),
    (1012, "C108", "Sara", "Delhi", "Chair", "Furniture", 8000.0, 2, "PENDING", "2026-01-16"),
]
columns = ["order_id", "customer_id", "customer_name", "city", "product", "category", "unit_price", "quantity", "status", "order_date"]
df_orders = spark.createDataFrame(orders_data, columns)
df_orders = (df_orders.withColumn("order_date", F.to_date("order_date"))
             .withColumn("order_amount", F.col("unit_price") * F.col("quantity")))
df_orders.show(truncate=False)
df_orders.printSchema()
```

The sample is an ecommerce order-line dataset. `order_amount` is calculated as `unit_price * quantity`.

# 2. Temporary views and `createOrReplaceTempView()`

A **temporary view** gives a DataFrame or query result a SQL name. It normally does not copy the data into a separate physical table.

```python
df_orders.createOrReplaceTempView("orders_view")
spark.sql("SELECT * FROM orders_view").show()
```

Mental model:

```text
DataFrame -> createOrReplaceTempView("orders_view")
          -> spark.sql("SELECT ... FROM orders_view")
          -> Result DataFrame
```

| Object | Meaning | Scope / access |
|---|---|---|
| DataFrame | Structured data and logical plan | Python variable such as `df_orders` |
| Local temporary view | SQL name for a DataFrame/query | Current `SparkSession` |
| Global temporary view | View in global temporary namespace | Same Spark application; use `global_temp.view_name` |
| Persisted table | Catalog table intended to outlive a session | Depends on catalog and storage configuration |

## Method definition and parameter

```python
df.createOrReplaceTempView(name)
```

| Parameter | Required | Meaning |
|---|---:|---|
| `name` | Yes | String identifier used in SQL queries |

Behavior:
- Creates a local temporary view, or replaces a view with the same name in the current session.
- It is not a persisted table and is not automatically cached.
- Spark evaluation is generally lazy until an action requires results.
- The view name is a SQL identifier, not a Python variable.

Related methods:

```python
df_orders.createTempView("orders_view")                  # Fails if name already exists
df_orders.createOrReplaceTempView("orders_view")         # Creates or replaces
df_orders.createOrReplaceGlobalTempView("orders_global")
spark.sql("SELECT * FROM global_temp.orders_global").show()
```

# 3. Temporary views — 10 examples

## Example 1 — Register a DataFrame

```python
df_orders.createOrReplaceTempView("orders_view")
spark.sql("SELECT * FROM orders_view").show()
```

**Explanation:** The view name can now be resolved by SQL in this session.

## Example 2 — View from a filtered DataFrame

```python
df_completed = df_orders.filter(F.col("status") == "COMPLETED")
df_completed.createOrReplaceTempView("completed_orders_view")
spark.sql("SELECT * FROM completed_orders_view").show()
```

**Use case:** A named logical stage containing completed orders only.

## Example 3 — View with selected columns

```python
df_orders.select("order_id", "customer_id", "city", "order_amount", "status") \
    .createOrReplaceTempView("order_summary_view")
spark.sql("SELECT * FROM order_summary_view").show()
```

**Explanation:** The view exposes only the selected columns.

## Example 4 — Replace a view definition

```python
df_orders.createOrReplaceTempView("orders_view")
df_orders.filter(F.col("order_amount") >= 30000).createOrReplaceTempView("orders_view")
spark.sql("SELECT order_id, order_amount FROM orders_view").show()
```

**Explanation:** The second registration replaces the earlier definition. Descriptive stage names help avoid accidental replacement.

## Example 5 — View with a derived column

```python
df_enriched = df_orders.withColumn(
    "order_band",
    F.when(F.col("order_amount") >= 50000, "HIGH")
     .when(F.col("order_amount") >= 20000, "MEDIUM")
     .otherwise("LOW")
)
df_enriched.createOrReplaceTempView("enriched_orders_view")
spark.sql("SELECT order_id, order_amount, order_band FROM enriched_orders_view").show()
```

**Explanation:** A derived column can be queried like any source column.

## Example 6 — Register multiple views

```python
df_orders.createOrReplaceTempView("orders_view")
df_customers = df_orders.select("customer_id", "customer_name", "city").distinct()
df_customers.createOrReplaceTempView("customers_view")
spark.sql("SELECT COUNT(*) AS order_rows FROM orders_view").show()
spark.sql("SELECT COUNT(*) AS customer_rows FROM customers_view").show()
```

**Use case:** Give different logical datasets different SQL names. In production, a customer view often comes from a customer master source.

## Example 7 — View used for aggregation

```python
df_orders.createOrReplaceTempView("orders_view")
spark.sql("SELECT city, SUM(order_amount) AS revenue FROM orders_view GROUP BY city").show()
```

**Explanation:** Views can be used in filtering, grouping, joins, and window queries.

## Example 8 — Register a joined DataFrame

```python
df_customer_city = df_orders.select("customer_id", "customer_name", "city").distinct()
df_joined = df_orders.join(df_customer_city,
    on=["customer_id", "customer_name", "city"], how="left")
df_joined.createOrReplaceTempView("orders_with_customer_view")
spark.sql("SELECT order_id, customer_name, city, order_amount FROM orders_with_customer_view").show()
```

**Explanation:** Any DataFrame, including a join result, can be registered. This join is illustrative because the sample orders already contain customer details.

## Example 9 — Global temporary view

```python
df_orders.createOrReplaceGlobalTempView("orders_global")
spark.sql("SELECT order_id, city, order_amount FROM global_temp.orders_global").show()
```

**Explanation:** Global temporary views use the `global_temp` namespace and can be shared across sessions in the same Spark application. They are still temporary.

## Example 10 — Inspect view names and query as a DataFrame

```python
df_orders.createOrReplaceTempView("orders_view")
print(spark.catalog.listTables())
df_city_counts = spark.sql("SELECT city, COUNT(*) AS order_count FROM orders_view GROUP BY city")
df_city_counts.printSchema()
df_city_counts.show()
```

**Explanation:** `spark.sql()` returns a DataFrame, so you can call `show()`, `filter()`, `write`, and other DataFrame methods on its result.

# 4. `spark.sql()` and 10 query examples

`spark.sql()` parses a Spark SQL statement and returns a DataFrame representing its result.

```python
result_df = spark.sql("SELECT * FROM orders_view")
```

The main parameter is the SQL query string (`sqlQuery`). Some Spark versions support optional parameter arguments; check your installed version's signature. Prefer supported parameter binding or validated values over unsafe string concatenation when queries use dynamic values.

Register the base view before running these examples:

```python
df_orders.createOrReplaceTempView("orders_view")
```

## Example 1 — Select all columns

```python
spark.sql("SELECT * FROM orders_view").show()
```

**Explanation:** `*` selects all view columns. Production code often benefits from explicitly selecting only needed fields.

## Example 2 — Select columns and aliases

```python
spark.sql("""
    SELECT order_id, customer_name AS customer, city,
           order_amount AS amount
    FROM orders_view
""").show()
```

**Explanation:** `AS` assigns output column names without changing the original view.

## Example 3 — Filter with `WHERE`

```python
spark.sql("""
    SELECT order_id, customer_name, city, order_amount
    FROM orders_view
    WHERE status = 'COMPLETED' AND order_amount >= 10000
""").show()
```

**Explanation:** `WHERE` filters rows; compound conditions use `AND`, `OR`, and `NOT`.

## Example 4 — Sort and limit

```python
spark.sql("""
    SELECT order_id, product, order_amount
    FROM orders_view
    ORDER BY order_amount DESC, order_id ASC
    LIMIT 5
""").show()
```

**Explanation:** `DESC` sorts descending, `ASC` ascending, and `LIMIT` caps returned rows. A tie-breaker helps make the selected top rows deterministic.

## Example 5 — Calculate an expression

```python
spark.sql("""
    SELECT order_id, unit_price, quantity,
           unit_price * quantity AS calculated_amount
    FROM orders_view
""").show()
```

**Explanation:** SQL arithmetic creates a derived output column; the source view remains unchanged.

## Example 6 — Conditional logic with `CASE WHEN`

```python
spark.sql("""
    SELECT order_id, order_amount,
        CASE WHEN order_amount >= 50000 THEN 'HIGH'
             WHEN order_amount >= 20000 THEN 'MEDIUM'
             ELSE 'LOW'
        END AS order_band
    FROM orders_view
""").show()
```

**Explanation:** Conditions are evaluated in order and the first matching branch wins.

## Example 7 — Group, aggregate, and filter groups

```python
spark.sql("""
    SELECT city, COUNT(*) AS order_count,
           SUM(order_amount) AS revenue,
           AVG(order_amount) AS average_amount
    FROM orders_view
    GROUP BY city
    HAVING SUM(order_amount) >= 20000
    ORDER BY revenue DESC
""").show()
```

**Explanation:** `GROUP BY` defines groups; `COUNT`, `SUM`, and `AVG` aggregate; `HAVING` filters groups after aggregation. `WHERE` filters rows before aggregation.

## Example 8 — Join two views

```python
customers_data = [
    ("C101", "Arun", "Gold"), ("C102", "Priya", "Silver"),
    ("C103", "Rahul", "Gold"), ("C104", "Meena", "Bronze"),
    ("C105", "Vikram", "Silver"), ("C106", "Divya", "Gold"),
    ("C107", "Kavin", "Bronze"), ("C108", "Sara", "Silver")
]
df_customers = spark.createDataFrame(customers_data,
    ["customer_id", "customer_name", "loyalty_tier"])
df_customers.createOrReplaceTempView("customers_view")

spark.sql("""
    SELECT o.order_id, o.customer_id, o.customer_name,
           c.loyalty_tier, o.order_amount
    FROM orders_view AS o
    LEFT JOIN customers_view AS c ON o.customer_id = c.customer_id
""").show()
```

**Explanation:** `LEFT JOIN` keeps all order rows and adds customer attributes where a key matches. Aliases `o` and `c` make references clearer.

## Example 9 — Common table expressions (CTEs)

```python
spark.sql("""
    WITH completed_orders AS (
        SELECT * FROM orders_view WHERE status = 'COMPLETED'
    ),
    city_revenue AS (
        SELECT city, SUM(order_amount) AS revenue
        FROM completed_orders GROUP BY city
    )
    SELECT city, revenue FROM city_revenue
    WHERE revenue > 20000 ORDER BY revenue DESC
""").show()
```

**Explanation:** `WITH` names query blocks within one SQL statement. CTEs improve readability but do not automatically guarantee caching/materialization.

## Example 10 — Window function

```python
spark.sql("""
    SELECT order_id, customer_id, order_date, order_amount,
        SUM(order_amount) OVER (
            PARTITION BY customer_id
            ORDER BY order_date, order_id
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS customer_running_total
    FROM orders_view
    ORDER BY customer_id, order_date, order_id
""").show()
```

**Explanation:** `PARTITION BY` resets the running total for each customer. The window order defines sequence; the frame includes previous rows and the current row.

# 5. SQL vs DSL — 10 paired examples

The **DataFrame DSL** uses Python methods like `select()`, `filter()`, `withColumn()`, `groupBy()`, and `join()`. **Spark SQL** expresses relational operations as SQL strings. Both are optimized by Spark SQL; SQL is not automatically faster than DSL, or vice versa.

Register the view first:

```python
df_orders.createOrReplaceTempView("orders_view")
```

## Pair 1 — Select columns

**DSL**
```python
df_orders.select("order_id", "customer_name", "order_amount").show()
```
**SQL**
```python
spark.sql("SELECT order_id, customer_name, order_amount FROM orders_view").show()
```
**Comparison:** DSL is convenient when a DataFrame variable is available; SQL is natural for relational query readers.

## Pair 2 — Filter rows

**DSL**
```python
df_orders.filter((F.col("status") == "COMPLETED") & (F.col("order_amount") > 10000)).show()
```
**SQL**
```python
spark.sql("SELECT * FROM orders_view WHERE status = 'COMPLETED' AND order_amount > 10000").show()
```
**Comparison:** DSL combines Spark column conditions with `&` and parentheses; SQL uses `AND`.

## Pair 3 — Add a calculated column

**DSL**
```python
df_orders.withColumn("discounted_amount", F.col("order_amount") * F.lit(0.90)) \
    .select("order_id", "order_amount", "discounted_amount").show()
```
**SQL**
```python
spark.sql("SELECT order_id, order_amount, order_amount * 0.90 AS discounted_amount FROM orders_view").show()
```
**Comparison:** `withColumn()` adds/replaces a column in the returned DataFrame; SQL `SELECT` defines the output columns. The discount is illustrative only.

## Pair 4 — Conditional classification

**DSL**
```python
df_orders.withColumn("order_band",
    F.when(F.col("order_amount") >= 50000, "HIGH")
     .when(F.col("order_amount") >= 20000, "MEDIUM")
     .otherwise("LOW")).select("order_id", "order_amount", "order_band").show()
```
**SQL**
```python
spark.sql("""
    SELECT order_id, order_amount,
        CASE WHEN order_amount >= 50000 THEN 'HIGH'
             WHEN order_amount >= 20000 THEN 'MEDIUM' ELSE 'LOW' END AS order_band
    FROM orders_view
""").show()
```
**Comparison:** DSL uses `when().otherwise()`; SQL uses `CASE WHEN`.

## Pair 5 — Group and aggregate

**DSL**
```python
df_orders.groupBy("city").agg(
    F.count("*").alias("order_count"),
    F.sum("order_amount").alias("revenue"),
    F.avg("order_amount").alias("avg_order_amount")
).show()
```
**SQL**
```python
spark.sql("""
    SELECT city, COUNT(*) AS order_count, SUM(order_amount) AS revenue,
           AVG(order_amount) AS avg_order_amount
    FROM orders_view GROUP BY city
""").show()
```
**Comparison:** DSL works well with Python-generated expressions; SQL is often readable for complex grouping logic.

## Pair 6 — Sort and limit

**DSL**
```python
df_orders.orderBy(F.col("order_amount").desc(), F.col("order_id").asc()).limit(5).show()
```
**SQL**
```python
spark.sql("SELECT * FROM orders_view ORDER BY order_amount DESC, order_id ASC LIMIT 5").show()
```
**Comparison:** Both perform a sort and limit; use tie-breakers for deterministic output.

## Pair 7 — Alias output columns

**DSL**
```python
df_orders.select(F.col("customer_name").alias("customer"), F.col("order_amount").alias("amount")).show()
```
**SQL**
```python
spark.sql("SELECT customer_name AS customer, order_amount AS amount FROM orders_view").show()
```
**Comparison:** Aliases change output names, not the source DataFrame in place.

## Pair 8 — Join

```python
df_customer_tiers = spark.createDataFrame(
    [("C101", "Gold"), ("C102", "Silver"), ("C103", "Gold"),
     ("C104", "Bronze"), ("C105", "Silver"), ("C106", "Gold"),
     ("C107", "Bronze"), ("C108", "Silver")],
    ["customer_id", "loyalty_tier"])
df_customer_tiers.createOrReplaceTempView("customer_tiers_view")
```
**DSL**
```python
df_orders.join(df_customer_tiers, on="customer_id", how="left") \
    .select("order_id", "customer_id", "loyalty_tier", "order_amount").show()
```
**SQL**
```python
spark.sql("""
    SELECT o.order_id, o.customer_id, c.loyalty_tier, o.order_amount
    FROM orders_view o LEFT JOIN customer_tiers_view c
      ON o.customer_id = c.customer_id
""").show()
```
**Comparison:** DSL exposes join keys/type as arguments; SQL can make multi-table joins easier to scan.

## Pair 9 — Filter nulls

**DSL**
```python
df_orders.filter(F.col("customer_id").isNotNull()).show()
```
**SQL**
```python
spark.sql("SELECT * FROM orders_view WHERE customer_id IS NOT NULL").show()
```
**Comparison:** DSL uses `isNull()` / `isNotNull()`; SQL uses `IS NULL` / `IS NOT NULL`. Do not compare with `= NULL`.

## Pair 10 — Rank orders with a window

**DSL**
```python
w = Window.partitionBy("customer_id").orderBy(F.col("order_amount").desc(), F.col("order_id").asc())
df_orders.withColumn("order_rank", F.row_number().over(w)) \
    .select("customer_id", "order_id", "order_amount", "order_rank").show()
```
**SQL**
```python
spark.sql("""
    SELECT customer_id, order_id, order_amount,
        ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_amount DESC, order_id ASC) AS order_rank
    FROM orders_view
""").show()
```
**Comparison:** DSL uses `Window.partitionBy()` and `Window.orderBy()`; SQL uses `OVER (...)`. `order_id` is a tie-breaker.

# 6. Realistic ETL: Bronze → Silver → Gold

## Bronze — register source

```python
bronze_orders = df_orders
bronze_orders.createOrReplaceTempView("bronze_orders")
```

## Silver — standardize and validate

```python
silver_orders = spark.sql("""
    SELECT order_id, customer_id,
           TRIM(customer_name) AS customer_name,
           INITCAP(TRIM(city)) AS city,
           product, category, unit_price, quantity,
           unit_price * quantity AS order_amount,
           UPPER(TRIM(status)) AS status,
           order_date
    FROM bronze_orders
    WHERE order_id IS NOT NULL AND customer_id IS NOT NULL
      AND unit_price IS NOT NULL AND quantity IS NOT NULL AND quantity > 0
""")
silver_orders.createOrReplaceTempView("silver_orders")
```

**Explanation:** `TRIM()` removes surrounding spaces, `INITCAP()` standardizes city display, and `UPPER()` standardizes status. This example filters null keys/prices/quantities and nonpositive quantities. Real pipelines should define duplicate, date, price, and allowed-status rules too.

## Gold — business metrics

```python
gold_city_revenue = spark.sql("""
    SELECT city, category,
           COUNT(*) AS order_line_count,
           COUNT(DISTINCT order_id) AS distinct_order_count,
           SUM(order_amount) AS gross_revenue,
           AVG(order_amount) AS average_line_amount
    FROM silver_orders
    WHERE status = 'COMPLETED'
    GROUP BY city, category
""")
gold_city_revenue.orderBy(F.col("gross_revenue").desc()).show(truncate=False)
```

**Explanation:** This is a Gold-style aggregate for reporting. If an order has several categories, it can be counted in more than one city/category group.

## Optional persistence

```python
# Requires a valid path and storage format in your environment.
# gold_city_revenue.write.mode("overwrite").format("parquet").save("/data/gold/city_revenue")
```

A temporary view is not a replacement for durable storage.

# 7. Large-data example and plan inspection

Generate data with Spark expressions rather than building a million Python tuples on the driver:

```python
large_orders = (
    spark.range(0, 1_000_000)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", F.concat(F.lit("C"), (F.col("order_id") % 100_000).cast("string")))
    .withColumn("city",
        F.when(F.col("order_id") % 5 == 0, "Chennai")
         .when(F.col("order_id") % 5 == 1, "Bengaluru")
         .when(F.col("order_id") % 5 == 2, "Mumbai")
         .when(F.col("order_id") % 5 == 3, "Hyderabad")
         .otherwise("Delhi"))
    .withColumn("unit_price", (F.col("order_id") % 5000 + 100).cast("double"))
    .withColumn("quantity", (F.col("order_id") % 5 + 1).cast("int"))
    .withColumn("order_amount", F.col("unit_price") * F.col("quantity"))
    .withColumn("status", F.when(F.col("order_id") % 10 == 0, "PENDING")
         .when(F.col("order_id") % 13 == 0, "CANCELLED").otherwise("COMPLETED"))
)
large_orders.createOrReplaceTempView("large_orders_view")
```

**SQL aggregation**
```python
large_city_metrics = spark.sql("""
    SELECT city, status, COUNT(*) AS row_count,
           SUM(order_amount) AS total_amount, AVG(order_amount) AS average_amount
    FROM large_orders_view GROUP BY city, status
""")
large_city_metrics.show()
```

**Equivalent DSL**
```python
large_city_metrics_dsl = large_orders.groupBy("city", "status").agg(
    F.count("*").alias("row_count"),
    F.sum("order_amount").alias("total_amount"),
    F.avg("order_amount").alias("average_amount"))
large_city_metrics_dsl.show()
```

**Compare plans**
```python
large_city_metrics.explain("formatted")
large_city_metrics_dsl.explain("formatted")
```

Inspect scans, projections, filters, aggregations, and exchange/shuffle operators. Physical plans vary by Spark version, source, configuration, statistics, and adaptive query execution. Compare equivalent logic; syntax alone does not determine speed.

# 8. Scope, lifecycle, mistakes, and performance

## Scope and lifecycle

- **Local view:** `df.createOrReplaceTempView("v")`; available in the current `SparkSession`; reference as `v`.
- **Global view:** `df.createOrReplaceGlobalTempView("v")`; reference as `global_temp.v`; temporary within the Spark application.
- **Persisted table:** `df.write.saveAsTable("database.table")`; requires suitable catalog/database/permissions and is designed for longer-lived access.
- **Drop local view:** `spark.catalog.dropTempView("orders_view")`.
- **Drop global view:** `spark.sql("DROP VIEW IF EXISTS global_temp.orders_global")` where supported.

## Common mistakes

1. **View not found:** register it in the current session before querying it.
2. **Global namespace omitted:** query `global_temp.orders_global`, not just `orders_global`.
3. **Assuming a view writes files:** use `.write` for Parquet/Delta/CSV or a table.
4. **Using a Python variable as a SQL table:** SQL cannot automatically resolve `df_orders`; register a view first.
5. **Comparing with null:** use `IS NULL` / `IS NOT NULL`, not `= NULL`.
6. **Confusing `WHERE` and `HAVING`:** `WHERE` filters rows before aggregation; `HAVING` filters groups after aggregation.
7. **Unsafe string concatenation:** avoid injecting untrusted values into SQL. Prefer supported parameter binding or validated values / DataFrame expressions.
8. **Nondeterministic top-N:** use `ORDER BY` with tie-breakers before `LIMIT`.
9. **Accidental view replacement:** re-registering the same name replaces its local view definition.
10. **Thinking `show()` saves data:** `show()` displays rows; it does not persist the result.

## Performance notes

1. SQL and DSL relational operations use Spark SQL's optimizer; equivalent operations often produce similar plans.
2. Registering a view does not automatically cache it.
3. Cache/persist only when a costly result is reused; an action is usually needed to materialize it.
4. Select only needed columns and filter early when logically valid.
5. Predicate pushdown depends on the source and expression.
6. Joins, `GROUP BY`, `DISTINCT`, and many windows can cause shuffles.
7. Global sorts can be expensive.
8. Inspect plans with `explain("formatted")`.
9. Benchmark equivalent workloads under comparable conditions.
10. Test nulls, duplicates, invalid dates, boundaries, and data-quality rules.

# 9. Practice assignments

Use `orders_view` and the sample data.

## Beginner
1. Register `df_orders` as `orders_view`.
2. Select `order_id`, `product`, and `order_amount` using SQL.
3. Find completed orders with amount above 15,000.
4. Sort orders by amount descending.
5. Calculate `amount_with_tax` using an illustrative tax rate of 18%.

## Intermediate
6. Calculate revenue and order count by city.
7. Show cities whose revenue is above 30,000.
8. Classify orders using `CASE WHEN`.
9. Create a customer-tier view and left-join it to orders.
10. Use a CTE to filter completed orders, then aggregate by category.

## Advanced
11. Calculate a running total per customer with a SQL window function.
12. Find each customer's latest completed order with `ROW_NUMBER()`.
13. Build Silver and Gold DataFrames using SQL and register them as views.
14. Implement an aggregation in both DSL and SQL, then compare formatted plans.
15. Explain why a temporary view is not a replacement for a persisted Delta/Parquet table.

### Challenge — Latest completed order per customer

```python
latest_completed = spark.sql("""
    WITH ranked_orders AS (
        SELECT *, ROW_NUMBER() OVER (
            PARTITION BY customer_id ORDER BY order_date DESC, order_id DESC
        ) AS rn
        FROM orders_view WHERE status = 'COMPLETED'
    )
    SELECT * FROM ranked_orders WHERE rn = 1
""")
latest_completed.show(truncate=False)
```

**Reasoning:** Filter eligible orders, rank each customer's rows newest-first, and keep rank 1. `order_id` is a tie-breaker.

# 10. Quick reference

| Goal | DataFrame DSL | Spark SQL |
|---|---|---|
| Register local view | `df.createOrReplaceTempView("v")` | Registration uses a DataFrame method |
| Run SQL | `spark.sql("SELECT ...")` | `spark.sql("SELECT ...")` |
| Select | `df.select("a", "b")` | `SELECT a, b FROM v` |
| Filter | `df.filter(F.col("a") > 10)` | `WHERE a > 10` |
| Add expression | `withColumn("x", ...)` | `expression AS x` in `SELECT` |
| Conditional | `F.when(...).otherwise(...)` | `CASE WHEN ... THEN ... END` |
| Aggregate | `groupBy(...).agg(...)` | `GROUP BY` |
| Filter groups | Filter aggregated DataFrame | `HAVING` |
| Join | `df1.join(df2, ...)` | `JOIN ... ON ...` |
| Window | `Window.partitionBy(...).orderBy(...)` | `OVER (PARTITION BY ... ORDER BY ...)` |
| Drop local view | `spark.catalog.dropTempView("v")` | `DROP VIEW IF EXISTS v` |
| Inspect plan | `df.explain("formatted")` | `spark.sql(...).explain("formatted")` |
| Persist | `df.write...` | CTAS / insert, depending on catalog |

## Final takeaway

- A **DataFrame** is structured data plus a logical query plan.
- A **temporary view** gives it a SQL name within a defined scope.
- **`spark.sql()`** runs SQL and returns a DataFrame.
- **SQL and DSL are two ways to express Spark SQL transformations**; choose based on clarity, maintainability, team familiarity, and how dynamically the pipeline is assembled.
- **Temporary does not mean cached or persisted.** Write to durable storage when the result must survive the session.
